In [ ]:
import os
import textwrap
import json

def list_subdirectories(flux_dir):
    return [
        name for name in os.listdir(flux_dir)
        if os.path.isdir(os.path.join(flux_dir, name)) and 'logs' not in name
    ]

def list_subdirectories2(filtered_flux_dir):
    return [
        name.removeprefix('filtered_') for name in os.listdir(filtered_flux_dir)
        if os.path.isdir(os.path.join(filtered_flux_dir, name)) and 'logs' not in name
    ]


# Get the set of subdirectories present in both lists
def get_concurrent_subdirectories(flux_dir, filtered_flux_dir):
    subdirs1 = set(list_subdirectories(flux_dir))
    subdirs2 = set(list_subdirectories2(filtered_flux_dir))
    return list(subdirs1 & subdirs2)


In [30]:
flux_results_path = '/data3/jhpark/testing_10'
filtered_flux_results_path = '/data3/jhpark/filtered_testing_10'
flux_results_dirs = list_subdirectories(flux_results_path)
filtered_flux_results_dirs = list_subdirectories2(filtered_flux_results_path)

exp_dirs = get_concurrent_subdirectories(flux_results_path, filtered_flux_results_path)


In [35]:
exp_dirs

['30c38878-292a-47b6-82b8-9990b5e17562',
 '7355f120-21d9-4ccf-8229-6cf89e3006fe',
 '2ace538d-aa56-42ed-975d-89033a22ce2c',
 'f89e9c76-c352-4eb4-9e8a-be5dad74eea5',
 '88cce569-8f4f-47f0-baab-0c16928dde8b',
 '5315be83-3c6b-4fa0-b6b4-c76ec0522c5d']

In [31]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def plot_comparison_png(parent_dir, subdirectory):
    comparison_path = os.path.join(parent_dir, subdirectory, "comparison.png")
    img = mpimg.imread(comparison_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"{subdirectory}/comparison.png")
    plt.show()


In [42]:
def plot_explanations(parent_dir, subdirectory):
    metadata_path = os.path.join(parent_dir, 'filtered_' + subdirectory, "metadata.json")
    print(metadata_path)
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    artifact_image_path = os.path.join(parent_dir, 'filtered_' + subdirectory, "artifact_image.png")
    img = mpimg.imread(artifact_image_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')

    for idx, artifact in enumerate(metadata):
        bbox = artifact['target_bbox']
        x1, y1, x2, y2 = bbox
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='r', facecolor='none')
        plt.gca().add_patch(rect)
        plt.gca().text(x1, y1 - 5, str(idx) +' ' + artifact['artifact_type'], color='white', fontsize=12, weight='bold', va='bottom', ha='left', bbox=dict(facecolor='r', alpha=0.5, edgecolor='none', pad=1))
        print('\n'.join(textwrap.wrap(artifact['explanation'], width=80)))
        print('--------------------------------')
    plt.show()

In [43]:
flux_results_iter = iter(exp_dirs)

In [50]:
flux_result = next(flux_results_iter)
plot_comparison_png(flux_results_path, flux_result)
plot_explanations(filtered_flux_results_path, flux_result)

StopIteration: 